In [4]:
import pandas as pd
import praw
import emoji
from datetime import datetime, timedelta
import time

# Connecting to reddit

reddit = praw.Reddit(
    client_id="k4C4H8PUXa0FGpBrdbZpQg", # My client(APP) ID
    client_secret="1Hv_M0wWh4LLYfpibEA5LGEm5lwH6Q", # secret key is password for connection
    user_agent="search_treasurer", # My Application name
)

In [12]:
# search term for reddit content/posts
search_term = "used cars"  

# Reading through all of the subreddits. 
subreddit = reddit.subreddit("all")

# Empty list to store data
data = []

# restrict the search to the last 2 days
two_days_ago = datetime.utcnow() - timedelta(days=2)
post_count = 0

for submission in subreddit.search(search_term, limit=None):
    if submission.created_utc >= two_days_ago.timestamp():
        author = submission.author.name if submission.author else "[deleted]"
        # Adding main post to data 
        data.append({
            "Content": submission.selftext,
            "Author": author,
            "Subreddit": submission.subreddit.display_name,
            "Score": submission.score,
            "NumComments": submission.num_comments,
            "Created": datetime.fromtimestamp(submission.created_utc),
            "URL": submission.url,
            "Type": "main post"
        })

        submission.comments.replace_more(limit=None)  # Load all comments
        for comment in submission.comments.list():
            comment_author = comment.author.name if comment.author else "[deleted]"

            # Add comment to data
            data.append({
                "Content": comment.body,
                "Author": comment_author,
                "Subreddit": submission.subreddit.display_name,
                "Score": comment.score,
                "NumComments": 0, # Comments don't have nested comments
                "Created": datetime.fromtimestamp(comment.created_utc),
                "URL": comment.permalink,
                "Type": "comment"
            })

        post_count += 1
        if post_count % 100 == 0:
            print(f"Processed {post_count} posts...")

        if post_count >= 10000:  
            break

    time.sleep(2)  # Sleep for 2 seconds between requests (adjust as needed)

# converting the data to a pandas dataframe
df = pd.DataFrame(data)
print(df.head())

                                             Content                Author  \
0  I drive a 2008 Camry XLE V6 that has served me...       sudden_cookie44   
1  Go drive a 2023 Camry v6.  Or sonata N line.  ...  Useful_Raspberry_500   
2  Don't give in to your colleagues. They sound l...  DeusExMachinaOverdue   
3  Well yeah. Your car was more expensive new.\n\...         DepthHour1669   
4  Guys at work sound like the type to pay thousa...                 hfusa   

           Subreddit  Score  NumComments             Created  \
0  whatcarshouldIbuy    191          217 2024-07-13 14:40:42   
1  whatcarshouldIbuy    116            0 2024-07-13 15:29:01   
2  whatcarshouldIbuy     85            0 2024-07-13 15:54:35   
3  whatcarshouldIbuy    544            0 2024-07-13 15:05:27   
4  whatcarshouldIbuy     51            0 2024-07-13 15:34:18   

                                                 URL       Type  
0  https://www.reddit.com/r/whatcarshouldIbuy/com...  main post  
1  /r/whatcars

In [13]:
df.head()

,Content,Author,Subreddit,Score,NumComments,Created,URL,Type
0,I drive a 2008 Camry XLE V6 that has served me...,sudden_cookie44,whatcarshouldIbuy,191,217,2024-07-13 14:40:42,https://www.reddit.com/r/whatcarshouldIbuy/com...,main post
1,Go drive a 2023 Camry v6. Or sonata N line. ...,Useful_Raspberry_500,whatcarshouldIbuy,116,0,2024-07-13 15:29:01,/r/whatcarshouldIbuy/comments/1e2lkll/they_don...,comment
2,Don't give in to your colleagues. They sound l...,DeusExMachinaOverdue,whatcarshouldIbuy,85,0,2024-07-13 15:54:35,/r/whatcarshouldIbuy/comments/1e2lkll/they_don...,comment
3,Well yeah. Your car was more expensive new.\n\...,DepthHour1669,whatcarshouldIbuy,544,0,2024-07-13 15:05:27,/r/whatcarshouldIbuy/comments/1e2lkll/they_don...,comment
4,Guys at work sound like the type to pay thousa...,hfusa,whatcarshouldIbuy,51,0,2024-07-13 15:34:18,/r/whatcarshouldIbuy/comments/1e2lkll/they_don...,comment


In [20]:
len(df)

380

In [21]:

# Get unique authors from the DataFrame
unique_authors = df['Author'].unique()

author_info = []
for author_name in unique_authors:
    if author_name != "[deleted]":
        try:
            redditor = reddit.redditor(author_name)
            author_info.append({
                "Author": redditor.name,
                "Link Karma": redditor.link_karma,
                "Comment Karma": redditor.comment_karma,
                "Is Employee": redditor.is_employee,
                "Is Mod": redditor.is_mod,
                "Account Created": datetime.fromtimestamp(redditor.created_utc)
            })
        except praw.exceptions.RedditAPIException:
            # Handle cases where the redditor might not exist or be inaccessible
            print(f"Could not fetch information for user: {author_name}")
            author_info.append({
                "Author": author_name,
                "Link Karma": None,
                "Comment Karma": None,
                "Is Employee": None,
                "Is Mod": None,
                "Account Created": None
            })
    else:
        author_info.append({
            "Author": author_name,
            "Link Karma": None,
            "Comment Karma": None,
            "Is Employee": None,
            "Is Mod": None,
            "Account Created": None
        })
    time.sleep(1)  # Sleep for 1 second between requests

# Create a new DataFrame with author information
author_df = pd.DataFrame(author_info)
print(author_df.head())


                 Author  Link Karma  Comment Karma Is Employee Is Mod  \
0       sudden_cookie44      1156.0        16299.0       False  False   
1  Useful_Raspberry_500       127.0        14386.0       False  False   
2  DeusExMachinaOverdue        15.0        17874.0       False  False   
3         DepthHour1669         1.0         1783.0       False  False   
4                 hfusa         1.0         1641.0       False  False   

      Account Created  
0 2021-01-28 08:51:30  
1 2024-05-06 10:32:50  
2 2018-11-12 15:45:49  
3 2024-06-09 18:02:06  
4 2023-04-05 11:31:30  


In [19]:


df['UniqueUserAttention'] = df[df['Type'] == 'comment']['Author'].nunique()

# 3. Merge with Author DataFrame
mega_df = pd.merge(df, author_df, on='Author', how='left')

mega_df.head()


,Content,Author,Subreddit,Score,NumComments,Created,URL,Type,UniqueUserAttention,Link Karma,Comment Karma,Is Employee,Is Mod,Account Created
0,I drive a 2008 Camry XLE V6 that has served me...,sudden_cookie44,whatcarshouldIbuy,191,217,2024-07-13 14:40:42,https://www.reddit.com/r/whatcarshouldIbuy/com...,main post,294,1121.0,16297.0,False,False,2021-01-28 08:51:30
1,Go drive a 2023 Camry v6. Or sonata N line. ...,Useful_Raspberry_500,whatcarshouldIbuy,116,0,2024-07-13 15:29:01,/r/whatcarshouldIbuy/comments/1e2lkll/they_don...,comment,294,127.0,14371.0,False,False,2024-05-06 10:32:50
2,Don't give in to your colleagues. They sound l...,DeusExMachinaOverdue,whatcarshouldIbuy,85,0,2024-07-13 15:54:35,/r/whatcarshouldIbuy/comments/1e2lkll/they_don...,comment,294,15.0,17861.0,False,False,2018-11-12 15:45:49
3,Well yeah. Your car was more expensive new.\n\...,DepthHour1669,whatcarshouldIbuy,544,0,2024-07-13 15:05:27,/r/whatcarshouldIbuy/comments/1e2lkll/they_don...,comment,294,1.0,1764.0,False,False,2024-06-09 18:02:06
4,Guys at work sound like the type to pay thousa...,hfusa,whatcarshouldIbuy,51,0,2024-07-13 15:34:18,/r/whatcarshouldIbuy/comments/1e2lkll/they_don...,comment,294,1.0,1638.0,False,False,2023-04-05 11:31:30


In [22]:
len(mega_df)

380

In [23]:
# Displaying hot posts and new posts related to used cars
subreddit = reddit.subreddit("usedcars")

# Get hot posts
print("Hot posts relevant to "+subreddit.display_name+":")
for submission in subreddit.hot(limit=10):
    print(submission.title, submission.url)
    
print("\n")
print("New posts relevant to "+subreddit.display_name+":")
# Get new posts
for submission in subreddit.new(limit=5):
    print(submission.title)


Hot posts relevant to usedcars:
[Guide] What used car should I get for what budget? https://www.reddit.com/r/UsedCars/comments/9ay6jf/guide_what_used_car_should_i_get_for_what_budget/
[Automod Update] Spam https://www.reddit.com/r/UsedCars/comments/1bkfwk6/automod_update_spam/
VIN question https://www.reddit.com/r/UsedCars/comments/1e3cxk7/vin_question/
What should I do?  https://www.reddit.com/r/UsedCars/comments/1e3ekdx/what_should_i_do/
New to Me Help https://www.reddit.com/r/UsedCars/comments/1e3dn9x/new_to_me_help/
Buy Used 2015 Genesis 5.0 W/ Issues? https://www.reddit.com/r/UsedCars/comments/1e3cgf6/buy_used_2015_genesis_50_w_issues/
Should I pull the trigger? https://www.reddit.com/r/UsedCars/comments/1e3cffj/should_i_pull_the_trigger/
Critique this buy with emphasis on added warranty https://www.reddit.com/r/UsedCars/comments/1e3bhah/critique_this_buy_with_emphasis_on_added_warranty/
2003 Nissan Frontier Crew Cab https://www.reddit.com/r/UsedCars/comments/1e3b4n9/2003_nissan_f

In [25]:
search_term = "donald trump"  

num_posts = 1000 # Number of hot posts to analyze

# reading through all of the subreddits.
subreddit = reddit.subreddit("all")
hot_posts = subreddit.search(search_term, sort="hot", limit=num_posts)

emoji_counter = {}
# using emoji.distinct_emoji_list() to get all the emojis in the post
# using emji_counter to count the frequency of each emoji
# and finally printing the top 10 emojis to quick look on latest sentiment. 
for submission in hot_posts:
    text = submission.title + " " + submission.selftext
    emojis = emoji.distinct_emoji_list(text)
    for em in emojis:
        if em in emoji_counter:
            emoji_counter[em] += 1
        else:
            emoji_counter[em] = 1

# Sort by frequency
sorted_emojis = sorted(emoji_counter.items(), key=lambda item: item[1], reverse=True)

print(f"Top 10 emojis used in hot posts about '{search_term}':")
for em, count in sorted_emojis[:10]:
    print(f"{em}: {count} times")


Top 10 emojis used in hot posts about 'donald trump':
👀: 1 times
😭: 1 times
💀: 1 times
⭐: 1 times


In [26]:
search_term = "keto diet"
subreddit = reddit.subreddit("all")

unique_usernames = set()
three_days_ago = datetime.utcnow() - timedelta(days=3)


# loop to interate through all of the posts and determine if the post was created in the last 3 days and if the author is not deleted
# further looping through to determine if the author has a username and identifying only unique authors
for submission in subreddit.search(search_term, limit=10000):
    if submission.created_utc >= three_days_ago.timestamp():  # Check post date
        author = submission.author
        if author and hasattr(author, 'name'):
            unique_usernames.add(author.name)

print(f"Unique usernames found for '{search_term}' in the last 3 days (up to 10,000 posts):")
for username in unique_usernames:
    print(username)


Unique usernames found for 'keto diet' in the last 3 days (up to 10,000 posts):
Sta-King00
Happy-Guy007
ExercisesForInjuries
